In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("temiz_veri/birlesik_veri.csv", index_col='datetime', parse_dates=True)
print("Veri boyutu:", df.shape)
df.head(3)

Veri boyutu: (34168, 10)


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,temperature_2m,relative_humidity_2m,precipitation
datetime,,,,,,,,,,
2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111,7.4,95,1.0
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667,6.7,93,0.8
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333,6.3,93,0.2


In [2]:
# Saat, gün, ay, yıl bilgileri
df['saat'] = df.index.hour
df['gun'] = df.index.dayofweek        # 0=Pazartesi, 6=Pazar
df['ay'] = df.index.month
df['yil'] = df.index.year
df['yilin_gunu'] = df.index.dayofyear

# Hafta sonu mu?
df['hafta_sonu'] = (df.index.dayofweek >= 5).astype(int)  # 1=hafta sonu, 0=hafta içi

# Mevsim
def mevsim_belirle(ay):
    if ay in [12, 1, 2]: return 0   # Kış
    elif ay in [3, 4, 5]: return 1  # İlkbahar
    elif ay in [6, 7, 8]: return 2  # Yaz
    else: return 3                   # Sonbahar

df['mevsim'] = df['ay'].apply(mevsim_belirle)

print("Eklenen özellikler:", ['saat','gun','ay','yil','yilin_gunu','hafta_sonu','mevsim'])
print("\nYeni boyut:", df.shape)

Eklenen özellikler: ['saat', 'gun', 'ay', 'yil', 'yilin_gunu', 'hafta_sonu', 'mevsim']

Yeni boyut: (34168, 17)


In [3]:
# 1 saat önceki tüketim
df['tuketim_lag_1'] = df['Global_active_power'].shift(1)

# 24 saat önceki tüketim (dün aynı saat)
df['tuketim_lag_24'] = df['Global_active_power'].shift(24)

# 168 saat önceki tüketim (geçen hafta aynı saat)
df['tuketim_lag_168'] = df['Global_active_power'].shift(168)

# Son 24 saatin ortalaması (hareketli ortalama)
df['tuketim_rolling_24'] = df['Global_active_power'].shift(1).rolling(window=24).mean()

print("Gecikmeli özellikler eklendi.")
print("\nEksik değer sayıları (lag'den kaynaklı):")
print(df[['tuketim_lag_1','tuketim_lag_24','tuketim_lag_168','tuketim_rolling_24']].isnull().sum())

Gecikmeli özellikler eklendi.

Eksik değer sayıları (lag'den kaynaklı):
tuketim_lag_1           1
tuketim_lag_24         24
tuketim_lag_168       168
tuketim_rolling_24     24
dtype: int64


In [4]:
# Lag'lerden kaynaklanan eksik değerleri sil
df_model = df.dropna()

print(f"Model verisi boyutu: {df_model.shape}")
print(f"\nTüm sütunlar:\n{df_model.columns.tolist()}")

# Kaydet
df_model.to_csv("temiz_veri/model_verisi.csv")
print("\nKaydedildi: temiz_veri/model_verisi.csv")

Model verisi boyutu: (34000, 21)

Tüm sütunlar:
['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'saat', 'gun', 'ay', 'yil', 'yilin_gunu', 'hafta_sonu', 'mevsim', 'tuketim_lag_1', 'tuketim_lag_24', 'tuketim_lag_168', 'tuketim_rolling_24']

Kaydedildi: temiz_veri/model_verisi.csv
